# 02 - Data Audit and Exploratory Analysis

**Purpose:** verify the acquired data and understand its classes, annotations, quality, diversity, imbalance, and leakage risks.

**Expected inputs:** immutable raw data and dataset metadata from stage 01.

**Expected outputs:** integrity checks, descriptive statistics, class distributions, image/annotation samples, duplicate checks, and a defensible split plan.

**Retain for the report:** dataset summary, class chart, representative samples, data-quality findings, imbalance evidence, and limitations.

In [1]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd / 'implementation', cwd, *cwd.parents]
IMPLEMENTATION_ROOT = next((p for p in candidates if p.name == 'implementation' and (p / 'stages').exists()), None)
if IMPLEMENTATION_ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or implementation directory.')
RAW_DATA = IMPLEMENTATION_ROOT / 'data' / 'raw'
FIGURES = IMPLEMENTATION_ROOT / 'outputs' / 'figures'
METRICS = IMPLEMENTATION_ROOT / 'outputs' / 'metrics'
SRC_ROOT = IMPLEMENTATION_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ipcv_attire.dataset_policy import build_manifest_bundle

## Audit and EDA

Validate the immutable source annotations and generate deterministic local manifests. This stage does not render arbitrary source images; qualitative review is restricted to the pending showcase candidates.

In [2]:
fashionpedia_root = RAW_DATA / 'fashionpedia'
annotation_root = fashionpedia_root / 'annotations'
manifest_dir = IMPLEMENTATION_ROOT / 'data' / 'interim' / 'manifests'
summary = build_manifest_bundle(
    train_annotations=annotation_root / 'instances_attributes_train2020.json',
    validation_annotations=annotation_root / 'instances_attributes_val2020.json',
    fashionpedia_root=fashionpedia_root,
    policy_path=IMPLEMENTATION_ROOT / 'data' / 'dataset-policy.json',
    output_dir=manifest_dir,
    showcase_manifest=IMPLEMENTATION_ROOT / 'data' / 'showcase-manifest.csv',
)
print(json.dumps(summary, indent=2))


{
  "policy_version": 3,
  "validation": {
    "train": {
      "images": 45623,
      "annotations": 333401,
      "categories": 46,
      "attributes": 294,
      "invalid_zero_area_boxes": 10
    },
    "validation": {
      "images": 1158,
      "annotations": 8781,
      "categories": 46,
      "attributes": 294,
      "invalid_zero_area_boxes": 0
    }
  },
  "manifest_images": 48823,
  "manifest_annotations": 342172,
  "excluded_invalid_annotations": 10,
  "unlabelled_official_test_images": 2042,
  "missing_image_files": 0,
  "relevant_images": 42125,
  "display_risk_images": 22267,
  "approved_showcase_images": 40,
  "candidate_showcase_images": 200,
  "compliance_labels": {
    "compliant": 1665,
    "non_compliant": 21011,
    "review_required": 24105
  },
  "target_support": {
    "allowed_sleeve": {
      "train": 22982,
      "internal_validation": 5381,
      "locked_in_domain_test": 713,
      "standalone_supported": true
    },
    "casual_or_tight_bottom": {
      "tra